# Lab 7: Building Gold-Layer Aggregates & Reporting Views
## Course: Data Stores & Pipelines (Trimester 2)

Welcome to Lab 7! In this notebook, we complete our **Medallion Architecture** pipeline by transitioning from the **Silver layer** (cleaned, conformed data) to the **Gold layer** (business-level aggregates and reporting-ready datasets).

### Learning Objectives:
* **Read Silver Data:** Ingest cleaned transactional data from the Silver storage path.
* **Aggregate & Model:** Compute business metrics such as total sales, transaction counts, and average order value grouped by department and date.
* **Load Gold Layer:** Persist the final aggregated tables for BI tools (like Tableau or Power BI).

### Step 1: Initialize SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, current_timestamp

spark = SparkSession.builder \
    .appName("Lab7_Gold_Layer_Aggregates") \
    .getOrCreate()

print(f"Spark Session Active. Version: {spark.version}")

### Step 2: Extract from Silver Layer
We read our cleaned and validated transactional data from the Silver layer storage path created in Lab 6.

In [ ]:
silver_path = "./lakehouse/silver/transactions"

# Read silver parquet data
silver_df = spark.read.parquet(silver_path)

print("--- Loaded Silver Data Preview ---")
silver_df.show(truncate=False)

### Step 3: Transform & Aggregate (Gold Layer Business Logic)
Next, we apply business-level transformations. We group transactions by department and transaction date to compute key performance indicators (KPIs): 
* Total revenue (`total_sales`)
* Total number of transactions (`transaction_count`)
* Average transaction amount (`avg_transaction_value`)

In [ ]:
# Compute departmental sales aggregates
gold_sales_summary = silver_df \
    .groupBy("dept", "txn_date") \
    .agg(
        spark_sum("amount").alias("total_sales"),
        count("txn_id").alias("transaction_count"),
        avg("amount").alias("avg_transaction_value")
    ) \
    .withColumn("updated_at", current_timestamp())

print("--- Gold Layer Aggregated Summary ---")
gold_sales_summary.show(truncate=False)

### Step 4: Load to Gold Storage
Finally, we write our aggregated Gold dataset to the target analytics path, making it immediately accessible for reporting dashboards.

In [ ]:
gold_path = "./lakehouse/gold/department_sales_summary"

gold_sales_summary.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(gold_path)

print(f"Gold Layer Pipeline complete! Aggregates successfully written to: {gold_path}")